In [1]:
import pandas as pd

columns = [
    "id", "label", "statement", "subject", "speaker",
    "job_title", "state_info", "party_affiliation",
    "barely_true_counts", "false_counts", "half_true_counts",
    "mostly_true_counts", "pants_fire_counts", "context"
]

train_df = pd.read_csv("data/train.tsv", sep = "\t", header=None, names = columns)
test_df = pd.read_csv("data/test.tsv", sep = "\t", header=None, names = columns, index_col=0)
cross_valid_df = pd.read_csv("data/valid.tsv", sep = "\t", header=None, names = columns, index_col=0)

print(train_df["label"].value_counts())

label
half-true      2114
false          1995
mostly-true    1962
true           1676
barely-true    1654
pants-fire      839
Name: count, dtype: int64


Columns names are not present with the dataset, so we create our own column names and assign it to our entries

In [2]:
label_map = {
    "pants-fire": 0, "false": 0, "barely-true": 0,
    "half-true": 1, "mostly-true": 1, "true": 1
}

train_df["binary_label"] = train_df["label"].map(label_map)
test_df["binary_label"] = test_df["label"].map(label_map)
cross_valid_df["binary_label"] = cross_valid_df["label"].map(label_map)

print(train_df["binary_label"].value_counts())

binary_label
1    5752
0    4488
Name: count, dtype: int64


We are collapsing 6 labels into 2 values- true or false, using .map

In [11]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

sample = train_df["statement"].iloc[0]
print(sample)

tokens = tokenizer(sample, padding = "max_length", truncation = True, max_length = 128)

print("\nKeys:", tokens.keys())
print("Input IDs:", tokens["input_ids"][:20])
print("Attention mask:", tokens["attention_mask"][:20])
print(tokenizer.convert_ids_to_tokens(tokens["input_ids"][:20]))


Says the Annies List political group supports third-trimester abortions on demand.

Keys: KeysView({'input_ids': [101, 2758, 1996, 8194, 2015, 2862, 2576, 2177, 6753, 2353, 1011, 12241, 20367, 11324, 2015, 2006, 5157, 1012, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1,

In [4]:
def tokenize(batch):
    return tokenizer(
        batch["statement"].tolist(),
        padding="max_length",
        truncation=True,
        max_length=128
    )

train_encodings = tokenize(train_df)
cross_valid_encodings   = tokenize(cross_valid_df)
test_encodings  = tokenize(test_df)

print("Input id:", train_encodings["input_ids"][1][:20]) 
print("Attention mask:", train_encodings["attention_mask"][1][:100]) 
print(len(train_encodings["input_ids"]))

Input id: [101, 2043, 2106, 1996, 6689, 1997, 5317, 2707, 1029, 2009, 2318, 2043, 3019, 3806, 2165, 2125, 2008, 2318, 2000, 4088]
Attention mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
10240


In [5]:
import torch
from torch.utils.data import Dataset

class FakeNewsDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

train_dataset = FakeNewsDataset(train_encodings, train_df["binary_label"].tolist())
val_dataset   = FakeNewsDataset(cross_valid_encodings,   cross_valid_df["binary_label"].tolist())
test_dataset  = FakeNewsDataset(test_encodings,  test_df["binary_label"].tolist())

print("Train size:", len(train_dataset))
print("Sample item keys:", train_dataset[0].keys())

Train size: 10240
Sample item keys: dict_keys(['input_ids', 'token_type_ids', 'attention_mask', 'labels'])


In [12]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels = 2
)

print(model.config.num_labels)

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 6107.91it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


2


We have loaded the bert model, now we train it on our dataset

In [16]:
from transformers import Trainer, TrainingArguments
import numpy as np
from sklearn.metrics import f1_score, accuracy_score

def compute_metrics(pred):
    labels = pred.label_ids
    preds = np.argmax(pred.predictions, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds)
    }

# training_args = TrainingArguments(
#     output_dir="./results",
#     num_train_epochs=2,
#     per_device_train_batch_size=16,
#     per_device_eval_batch_size=16,
#     eval_strategy="epoch",
#     save_strategy="epoch",
#     load_best_model_at_end=True,
#     logging_dir="./logs",
# )

# New training function with tuned parameters

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    logging_dir="./logs",
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_steps=100,
)

trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics = compute_metrics,
)

print("Trainer is ready ")

[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Trainer is ready 


In [17]:
trainer.train()

/home/kamalesh/.local/lib/python3.14/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.667206,0.658083,0.628505,0.704644
2,0.615187,0.642631,0.644860,0.669565


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.19it/s]
/home/kamalesh/.local/lib/python3.14/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.11it/s]


TrainOutput(global_step=1280, training_loss=0.6304570436477661, metrics={'train_runtime': 1709.0152, 'train_samples_per_second': 11.984, 'train_steps_per_second': 0.749, 'total_flos': 678233081118720.0, 'train_loss': 0.6304570436477661, 'epoch': 2.0})

From the results, we notice that training loss decreasing from epoch 1 to 3, but validation loss increseases from epoch 1 to 3. This is a case of overfitting, so epoch 1 is the ideal model. Although, we can fine tune the model further.

In [18]:
results = trainer.evaluate(test_dataset)
print(results)

/home/kamalesh/.local/lib/python3.14/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Accuracy,F1
0.615187,0.643628,2,0.651144,0.695592


{'eval_loss': 0.6436276435852051, 'eval_accuracy': 0.6511444356748224, 'eval_f1': 0.6955922865013774}


{'eval_loss': 0.661439836025238, 'eval_accuracy': 0.5966850828729282, 'eval_f1': 0.718457300275482}
There are the results we get when we run our model on test dataset. We notice that there is a 59% accuracy, which isn't bad since even most research published papers get 65-70% accuracy for LIAR dataset. This outcome is due to the tricky nature of real vs fake news headlines. Now it time to tune it further and check results.

Now after we tune the parameter, this is the result
{'eval_loss': 0.6436276435852051, 'eval_accuracy': 0.6511444356748224, 'eval_f1': 0.6955922865013774}
Accuracy has jumped up to 65 % on test dataset, while loss has only decrease a bit, we chose this model.

In [19]:
trainer.save_model("./fake_news_model")
tokenizer.save_pretrained("./fake_news_model")
print("Model saved!")

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  4.72it/s]

Model saved!


In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np

predictions = trainer.predict(test_dataset)
preds = np.argmax(predictions.predictions, axis=1)
labels = predictions.label_ids

cm = confusion_matrix(labels, preds)
print("Confusion Matrix:")
print(f"                 Predicted FAKE  Predicted REAL")
print(f"Actual FAKE      {cm[0][0]}            {cm[0][1]}")
print(f"Actual REAL      {cm[1][0]}            {cm[1][1]}")

/home/kamalesh/.local/lib/python3.14/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Confusion Matrix:
                 Predicted FAKE  Predicted REAL
Actual FAKE      320            233
Actual REAL      209            505

Classification Report:
              precision    recall  f1-score   support

        FAKE       0.60      0.58      0.59       553
        REAL       0.68      0.71      0.70       714

    accuracy                           0.65      1267
   macro avg       0.64      0.64      0.64      1267
weighted avg       0.65      0.65      0.65      1267



In [22]:
import os
print(os.path.abspath("./fake_news_model"))

/home/kamalesh/Desktop/Projects/Fake News Detection/Fake-News-Detection-Transformers-/fake_news_model
